In [ ]:
%pwd

In [ ]:
import os
os.chdir("../")

In [ ]:
%pwd

### Loading the Book

In [ ]:
from langchain.document_loaders import PyPDFLoader, DirectoryLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

In [ ]:
def load_pdf_files(data):
    loader = DirectoryLoader(
        data,
        glob="*.pdf",
        loader_cls=PyPDFLoader,
    )
    
    documents = loader.load()
    return documents    

In [ ]:
extracted_data = load_pdf_files("data")

In [ ]:
extracted_data

In [ ]:
len(extracted_data)

### Removing unwanted data from the book


In [ ]:
from typing import List
from langchain.schema import Document
def filter_to_minimal_docs(docs:List[Document]) -> List[Document]:
    
    minimal_docs: List[Document] = []
    for doc in docs:
        src = doc.metadata.get("source")
        minimal_docs.append(
            Document(
                page_content=doc.page_content,
                metadeta={"source": src,}
            )
        )
    return minimal_docs

In [ ]:
minimal_docs = filter_to_minimal_docs(extracted_data)

In [ ]:
minimal_docs 

### Chunking


In [ ]:
def text_split(minimal_docs):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=20,
    )
    
    text_chunk = text_splitter.split_documents(minimal_docs)
    return text_chunk

In [ ]:
text_chunk = text_split(minimal_docs)
print(f"Number of text chunks: {len(text_chunk)}")

In [ ]:
text_chunk

### Embedding

In [ ]:
from langchain.embeddings import HuggingFaceEmbeddings

def download_embeddings():
    model_name = "sentence-transformers/all-MiniLM-L6-v2"
    embeddings = HuggingFaceEmbeddings(
        model_name=model_name,
    )
    return embeddings

embedding = download_embeddings()

In [ ]:
embedding

In [ ]:
vector = embedding.embed_query("Hello world")
vector

In [ ]:
print(f"Vector length: {len(vector)}")

### Pinecone store

In [ ]:
from dotenv import load_dotenv
import os
load_dotenv()

In [ ]:
import os

PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
GROQ_API_KEY = os.getenv("GROQ_API_KEY")

In [ ]:
from pinecone import Pinecone
pinecone_api_key = PINECONE_API_KEY

pc = Pinecone(api_key=pinecone_api_key)

In [ ]:
pc

In [ ]:
from pinecone import ServerlessSpec

index_name = "baymax-index"

if not pc.has_index(index_name):
    pc.create_index(
        name=index_name,
        dimension=384,  
        metric="cosine",  
        spec=ServerlessSpec(cloud ="aws", region="us-east-1")
    )
    
index = pc.Index(index_name)

In [ ]:
from langchain_pinecone import PineconeVectorStore
docsearch = PineconeVectorStore.from_documents(
    documents=text_chunk,
    embedding=embedding,
    index_name=index_name,
)

In [ ]:
#load existing index
from langchain_pinecone import PineconeVectorStore
docsearch = PineconeVectorStore.from_existing_index(
    index_name =index_name,
    embedding=embedding
)

### Retriever Setup


In [ ]:
retriever = docsearch.as_retriever(
    search_type="similarity",
    search_kwargs={
        "k": 3,
    }
)

In [ ]:
retrieved_docs = retriever.invoke("What is acne")
retrieved_docs

In [ ]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq
load_dotenv()

llm = ChatGroq(
    model_name="llama3-70b-8192",
    temperature=0.7
)

In [ ]:
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

In [ ]:
system_prompt = (
    "You are a medical assistant for question-answering tasks. "
    "Use the following pieces of retrieved context to answer the question. "
    "If you don't know the answer, say you don't know. "
    "Use 3 sentences maximum and be concise.\n\n"
    "{context}"
)

prompt = ChatPromptTemplate.from_messages(
    [ 
        ("system", system_prompt),
        ("human", "{question}"),
    ]
)

In [ ]:
question_answer_chain = create_stuff_documents_chain (ChatGroq, prompt)
rag_chain = create_retrieval_chain(retriever, question_answer_chain)